In [13]:
import numpy as np
import pandas as pd
import random

In [14]:
def best_var(X, y):
    best_score, best_col, best_thresh = np.inf, None, None

    for col in range(X.shape[1]):
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, y, col, thresh)
            if score < best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, y, feature_column, threshold):
    true_array = y[X[:, feature_column] >= threshold]
    false_array = y[X[:, feature_column] < threshold]

    n1, n2 = len(true_array), len(false_array)
    true_pct, false_pct = n1/(n1+n2), n2/(n1+n2)
    score = true_pct * np.var(true_array) + false_pct * np.var(false_array)

    return score

In [15]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: float

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None

In [40]:
def _build(X, y, depth = 2, score_cutoff = 0):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_var(X, y)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or np.var(y) - best_score <= score_cutoff:
        mean = np.average(y)
        return LeafNode(mean)
    else:

        left_x, left_y = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh]
        right_x, right_y = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh]
        curr.leftNode = _build(left_x, left_y, depth-1, score_cutoff)
        curr.rightNode = _build(right_x, right_y, depth-1, score_cutoff)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

def create_gradient_boost(X, y, depth, num_trees = 3, lr = 0.1):
    ret = [np.average(y)]
    for _ in range(num_trees):
        new_y = y - gradient_boost_predict(X,ret)
        ret.append(_build(X, new_y, depth = depth))
    return ret

def gradient_boost_predict(X, f, lr = 0.1):
    curr = np.zeros(X.shape[0])
    for tree in f[1:]:
        curr += predict(X, tree)
    return f[0] + lr * curr 

In [38]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from collections import deque

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [41]:
X_rt = np.array([
    [1.0, 5.0],
    [2.0, 3.0],
    [3.0, 4.0],
    [4.0, 1.0],
    [5.0, 2.0],
    [6.0, 6.0],
])
y_rt = np.array([1.0, 2.0, 1.5, 9.0, 8.5, 10.0])

LEARNING_RATE = 0.1
local_gb = create_gradient_boost(X_rt, y_rt, depth = 3)
res = gradient_boost_predict(X_rt, local_gb)
print(res)
_display_graph(local_gb[1])

[4.159  4.43   4.2945 6.327  6.1915 6.598 ]
col: 0, threshold: 3.50 | 
col: 0, threshold: 5.50 | col: 0, threshold: 1.50 | 
res: 4.666666666666667 | col: 0, threshold: 4.50 | col: 0, threshold: 2.50 | res: -4.333333333333333 | 
res: 3.166666666666667 | res: 3.666666666666667 | res: -3.833333333333333 | res: -3.333333333333333 | 


In [47]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(n_samples=300, n_features=5, noise=10.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

gbr = GradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    criterion="squared_error",
    random_state=0,
)
gbr.fit(X_train, y_train)
local_gb = create_gradient_boost(X_train, y_train, num_trees=100, depth = 3)

res = gradient_boost_predict(X_test, local_gb)
sk_preds = gbr.predict(X_test)
# print(sk_preds)
# print(res)
print(mean_squared_error(y_test, sk_preds), r2_score(y_test, sk_preds))
print(mean_squared_error(y_test, res), r2_score(y_test, res))

463.8335084601976 0.9582887008684305
467.4389053449999 0.9579644772295449
